In [ ]:
import os
import random
import hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from tensorflow.keras import layers, Model
from PIL import Image
import imagehash

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
SSL_EPOCHS = 30
CLASSIFIER_EPOCHS = 30
TEMPERATURE = 0.5
AUTOTUNE = tf.data.AUTOTUNE
COMBINED_PATH = "/kaggle/input/datasets/sabbir4724/rose-leaf"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

image_paths = []
labels = []
class_names = sorted([
    d for d in os.listdir(COMBINED_PATH)
    if os.path.isdir(os.path.join(COMBINED_PATH, d))
])
class_to_idx = {cls: i for i, cls in enumerate(class_names)}

for cls in class_names:
    folder = os.path.join(COMBINED_PATH, cls)
    for img in sorted(os.listdir(folder)):
        path = os.path.join(folder, img)
        if os.path.isfile(path):
            image_paths.append(path)
            labels.append(class_to_idx[cls])

image_paths = np.array(image_paths)
labels = np.array(labels)
num_classes = len(class_names)

print("Classes:", num_classes)
print("Total images:", len(image_paths))
print("Class names:", class_names)
print("\nClass distribution:")
print(pd.Series(labels).map({v:k for k,v in class_to_idx.items()}).value_counts().sort_index())

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("Train      :", len(train_paths))
print("Validation :", len(val_paths))
print("Test       :", len(test_paths))
print("Total      :", len(train_paths) + len(val_paths) + len(test_paths))
print("\nExpected proportions:")
print("Train      :", len(train_paths) / len(image_paths))
print("Validation :", len(val_paths) / len(image_paths))
print("Test       :", len(test_paths) / len(image_paths))

train_set = set(train_paths)
val_set = set(val_paths)
test_set = set(test_paths)

print("\nPath overlap:")
print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test      :", len(train_set & test_set))
print("Validation ∩ Test :", len(val_set & test_set))

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(
        img, channels=3, expand_animations=False
    )
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
).map(load_image, num_parallel_calls=AUTOTUNE)

train_ds = (
    train_ds
    .shuffle(len(train_paths), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
).map(load_image, num_parallel_calls=AUTOTUNE)

val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_paths, test_labels)
).map(load_image, num_parallel_calls=AUTOTUNE)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Datasets created successfully.")

def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def perceptual_hash(path):
    with Image.open(path) as img:
        return imagehash.phash(img)

def duplicate_audit(paths_a, paths_b):
    md5_a = {}
    for p in paths_a:
        md5_a.setdefault(md5_file(p), []).append(p)

    md5_b = {}
    for p in paths_b:
        md5_b.setdefault(md5_file(p), []).append(p)

    common_md5 = set(md5_a) & set(md5_b)
    byte_identical_pairs = [
        (pa, pb)
        for h in common_md5
        for pa in md5_a[h]
        for pb in md5_b[h]
    ]

    hashes_a = [(p, perceptual_hash(p)) for p in paths_a]
    hashes_b = [(p, perceptual_hash(p)) for p in paths_b]

    near_pairs = []
    affected_b = set()
    for pa, ha in hashes_a:
        for pb, hb in hashes_b:
            dist = ha - hb
            if dist <= 5:
                near_pairs.append((pa, pb, dist))
                affected_b.add(pb)

    return byte_identical_pairs, near_pairs, affected_b

pairs = [
    ("Train-Validation", train_paths, val_paths),
    ("Train-Test", train_paths, test_paths),
    ("Validation-Test", val_paths, test_paths)
]

audit_rows = []
for name, a, b in pairs:
    exact_path_overlap = len(set(a) & set(b))
    byte_pairs, near_pairs, affected_b = duplicate_audit(a, b)
    audit_rows.append({
        "Comparison": name,
        "Exact path overlap": exact_path_overlap,
        "Byte-identical MD5 pairs": len(byte_pairs),
        "Near-duplicate pairs (pHash <= 5)": len(near_pairs),
        "Affected images in second partition": len(affected_b),
        "Affected percentage": (
            100 * len(affected_b) / len(b) if len(b) else 0
        )
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
audit_df.to_csv("three_way_split_integrity_audit.csv", index=False)
print("\nAudit saved: three_way_split_integrity_audit.csv")

augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1)
])

base_encoder = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_encoder.trainable = True

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_encoder(x, training=True)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
projection = layers.Dense(128)(x)

simclr_model = Model(inputs, projection)
simclr_model.summary()

def nt_xent_loss(zis, zjs, temperature=0.5):
    zis = tf.math.l2_normalize(zis, axis=1)
    zjs = tf.math.l2_normalize(zjs, axis=1)
    representations = tf.concat([zis, zjs], axis=0)
    similarity_matrix = tf.matmul(
        representations,
        representations,
        transpose_b=True
    )
    batch_size = tf.shape(zis)[0]
    mask = tf.eye(2 * batch_size)
    similarity_matrix = similarity_matrix - mask * 1e9
    positives = tf.concat([
        tf.range(batch_size, 2 * batch_size),
        tf.range(0, batch_size)
    ], axis=0)
    logits = similarity_matrix / temperature
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        positives,
        logits,
        from_logits=True
    )
    return tf.reduce_mean(loss)

optimizer = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(images):
    aug1 = augmenter(images)
    aug2 = augmenter(images)
    with tf.GradientTape() as tape:
        z1 = simclr_model(aug1, training=True)
        z2 = simclr_model(aug2, training=True)
        loss = nt_xent_loss(z1, z2, TEMPERATURE)
    grads = tape.gradient(
        loss,
        simclr_model.trainable_variables
    )
    optimizer.apply_gradients(
        zip(grads, simclr_model.trainable_variables)
    )
    return loss

ssl_losses = []
for epoch in range(SSL_EPOCHS):
    batch_losses = []
    for images, _ in train_ds:
        loss = train_step(images)
        batch_losses.append(loss.numpy())
    epoch_loss = np.mean(batch_losses)
    ssl_losses.append(epoch_loss)
    print(
        f"Epoch {epoch + 1}/{SSL_EPOCHS} "
        f"Loss={epoch_loss:.4f}"
    )

plt.figure(figsize=(8, 5))
plt.plot(ssl_losses)
plt.title("SimCLR NT-Xent Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.savefig("simclr_loss.png", dpi=600)
plt.show()

base_encoder.save("ssl_encoder.keras")
print("SSL encoder saved.")

feature_extractor = tf.keras.models.load_model(
    "ssl_encoder.keras"
)

class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)

    shared_dense_1 = layers.Dense(
        channel // ratio,
        activation="relu"
    )
    shared_dense_2 = layers.Dense(channel)

    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))

    channel_attention = layers.Add()([avg_out, max_out])
    channel_attention = layers.Activation(
        "sigmoid"
    )(channel_attention)

    channel_refined = layers.Multiply()([
        input_feature,
        channel_attention
    ])

    avg_pool_spatial = ChannelMean()(channel_refined)
    max_pool_spatial = ChannelMax()(channel_refined)
    concat = layers.Concatenate(axis=-1)([
        avg_pool_spatial,
        max_pool_spatial
    ])

    spatial_attention = layers.Conv2D(
        filters=1,
        kernel_size=7,
        padding="same",
        activation="sigmoid"
    )(concat)

    refined_feature = layers.Multiply()([
        channel_refined,
        spatial_attention
    ])

    return refined_feature

NUM_CLASSES = len(class_names)

inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)
x = tf.keras.applications.mobilenet_v2.preprocess_input(
    inputs
)
x = feature_extractor(x, training=False)
x = cbam_block(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(
    512,
    activation="relu",
    name="feature_dense"
)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    mode="min",
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    mode="min",
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CLASSIFIER_EPOCHS,
    callbacks=[
        checkpoint,
        early_stop,
        reduce_lr
    ]
)

model.load_weights("best_model.weights.h5")
print("Best validation-loss checkpoint loaded.")

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("accuracy_curve.png", dpi=600)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=600)
plt.show()

test_loss, test_acc = model.evaluate(
    test_ds,
    verbose=1
)

print("\n==============================")
print("FINAL UNTOUCHED TEST RESULT")
print("==============================")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

y_true = []
y_pred = []
y_prob = []

for images, labels_batch in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels_batch.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_prob.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print("Test prediction completed.")
print("Test samples:", len(y_true))

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
)
print(report)

with open(
    "test_classification_report.txt",
    "w"
) as f:
    f.write(report)

test_acc = accuracy_score(y_true, y_pred)
test_precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)
test_recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)
test_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("==============================")
print("FINAL TEST METRICS")
print("==============================")
print(f"Accuracy  : {test_acc:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1 Score  : {test_f1:.4f}")

metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Value": [
        test_acc,
        test_precision,
        test_recall,
        test_f1
    ]
})

metrics_df.to_csv(
    "final_test_metrics.csv",
    index=False
)

cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(10, 8))
plt.imshow(cm)
plt.colorbar()
plt.xticks(
    np.arange(len(class_names)),
    class_names,
    rotation=90
)
plt.yticks(
    np.arange(len(class_names)),
    class_names
)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Confusion Matrix — Untouched Test Set")
plt.tight_layout()
plt.savefig(
    "test_confusion_matrix.png",
    dpi=600
)
plt.show()

n_classes = len(class_names)
y_true_bin = label_binarize(
    y_true,
    classes=np.arange(n_classes)
)

plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(
        y_true_bin[:, i],
        y_prob[:, i]
    )
    roc_auc = auc(fpr, tpr)
    plt.plot(
        fpr,
        tpr,
        label=f"{class_names[i]} (AUC={roc_auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-Class ROC Curve — Untouched Test Set")
plt.legend(
    bbox_to_anchor=(1.05, 1)
)
plt.tight_layout()
plt.savefig(
    "test_roc_curve.png",
    dpi=300
)
plt.show()

feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("feature_dense").output
)

features = feature_model.predict(
    test_ds,
    verbose=1
)
print("Feature Shape:", features.shape)

perplexity = min(30, max(5, (len(features) - 1) // 3))

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    random_state=SEED,
    init="pca"
)
features_2d = tsne.fit_transform(features)

plt.figure(figsize=(15, 10))
for i, class_name in enumerate(class_names):
    idx = np.where(y_true == i)[0]
    plt.scatter(
        features_2d[idx, 0],
        features_2d[idx, 1],
        label=class_name,
        alpha=0.7
    )

plt.title(
    "t-SNE Visualization of Test Feature Embeddings"
)
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=8
)
plt.tight_layout()
plt.savefig(
    "test_tsne_class_names.png",
    dpi=600
)
plt.show()

model.save(
    "AgriSSL_CBAMNet_final_test_evaluated.keras"
)
print("Final model saved.")
print("Test set was used only for final evaluation.")

print("================================================")
print("FINAL EXPERIMENT SUMMARY")
print("================================================")
print(f"Total images       : {len(image_paths)}")
print(f"Training images    : {len(train_paths)}")
print(f"Validation images  : {len(val_paths)}")
print(f"Test images        : {len(test_paths)}")
print("\nModel-selection set : Validation")
print("Final evaluation set: Untouched Test")
print("\nDuplicate integrity:")
display(audit_df)
print("\nFinal test metrics:")
display(metrics_df)